# Project 2: retrieval and T5 scoring

Run on an A40 or A30 Jupyter session, with this folder as the working directory. Run the cells in order.

In [ ]:
from utils import env

env.setup()

In [ ]:
import os
import shutil

import pandas as pd

from utils import config
import retrieval

In [ ]:
dev = retrieval.read_jsonl(retrieval.DATA_PATH)
print(len(dev), "dev questions")

## Step 1: top 15 BM25 passages for every dev question
Saved to `bm25_top15.json` in `DIR_OUTPUTS`. Questions already in the cache are skipped.

In [ ]:
searcher = retrieval.open_searcher(retrieval.INDEX)
cache = retrieval.build_cache(dev, searcher, threads=8)

## Step 2: score all 15 passages per question with monoT5
Saved to `t5_scores.jsonl` in `DIR_OUTPUTS`. Finished questions are skipped if you rerun this cell.

In [ ]:
retriever = retrieval.Retriever(cache=cache, searcher=searcher)
scorer = retrieval.T5Scorer()
retrieval.score_questions(dev, retriever, scorer)

## Step 3: threshold and cap sweep
Replays adaptive stopping on the saved scores, so no model runs here. `gold_in_bm25_topkeep` is the plain BM25 top-2 baseline.

In [ ]:
records = retrieval.read_jsonl(retrieval.SCORES_PATH)
rows = retrieval.sweep(records)
retrieval.write_sweep_csv(rows)
pd.DataFrame(rows)

In [ ]:
archive = shutil.make_archive(os.path.join(config.ROOT, "project2_retrieval_results"), "zip",
                              os.path.dirname(config.DIR_OUTPUTS), os.path.basename(config.DIR_OUTPUTS))
print("Done. Download", archive)